## Setup

In [1]:
# Goal: Take an external dataset, and parse it down to a dataset with format
# description | resume_accept | resume_reject
# job description, an example of a good resume, an example of a bad resume 

In [2]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
HOME_DIRECTORY = Path('..')
DATASET_PATH = HOME_DIRECTORY / "data" / "dataset"

In [4]:
# Hugging Face token needed for some gated datasets
try: 
    with open(HOME_DIRECTORY / 'config.json') as configfile:
        cfg = json.load(configfile)
        os.environ['HF_TOKEN'] = cfg['HF_TOKEN']
    print('Token found')
    HAS_TOKEN = True
except:
    print('Token NOT FOUND')
    HAS_TOKEN = False

Token found


In [5]:
# Define our datasets

# Dataset 1: 
dataset_1_name = "AzharAli05"
dataset_1_url = "hf://datasets/AzharAli05/Resume-Screening-Dataset/dataset.csv"

# Dataset 2: Split into train/test.
dataset_2_name = "cnamuangtoun"
dataset_2_url_1 = "hf://datasets/cnamuangtoun/resume-job-description-fit/train.csv"
dataset_2_url_2 = "hf://datasets/cnamuangtoun/resume-job-description-fit/test.csv"

# Dataset 3: 
dataset_3_name = "MikePfunk28"
dataset_3_url = "hf://datasets/MikePfunk28/resume-training-dataset/training_data.jsonl"

# Other datasets:
# datasetmaster/resumes: "hf://dataset/datasetmaster/resumes/master_resumes.jsonl"

## Dataset Prep

### Dataset 1
https://huggingface.co/datasets/AzharAli05/Resume-Screening-Dataset

In [6]:
df_1_raw = pd.read_csv(dataset_1_url)
print(df_1_raw.shape)
df_1_raw.head()

c:\Users\dyaom\Documents\School\2025-2026\DSC 261\ScalableOversightProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(10174, 5)


,Role,Resume,Decision,Reason_for_decision,Job_Description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [7]:
# I'll process the whole dataset down to a subset with a known format
# we don't mind losing a bunch of resumes, we don't need that many
df_1 = df_1_raw.copy()
df_1.columns = ['role', 'resume', 'decision', 'reasoning', 'description']
df_1['decision'] = (df_1['decision'] == "select")
print(df_1.shape)
df_1.head()

(10174, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,True,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [8]:
# Only extract resumes with a known format
# "Here" as a prefix indicates that this a LLM-generated resume ("Here's a professional resume for...")
# while "Available upon request" indicates that this resume has a references section, and so we know where it ends
prefix = "Here"
suffix = "Available upon request."
df_1 = df_1[df_1['resume'].str.startswith(prefix)]
df_1 = df_1[df_1['resume'].str.contains(suffix)]
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,"Here's a sample resume for Jose Hall, a skille...",False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [9]:
# Remove the "Here's a resume!" prefix
df_1['resume'] = df_1['resume'].str.split(':\n\n', n=1).str[1]
df_1 = df_1[~df_1['resume'].isna()]
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Patrick Mcclain\nHuman Resources Specialist\n\...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [10]:
# and remove anything after the references (this is usually LLM commentary)
df_1['resume'] = df_1['resume'].str.rsplit(suffix, n=1).str[0] + suffix
print(df_1.shape)
df_1.head()

(6067, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
2,Human Resources Specialist,Patrick Mcclain\nHuman Resources Specialist\n\...,False,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
5,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...


In [11]:
# Explore: how many unique positions are there?
df_1_resume_counts = df_1.groupby('description')['resume'].count().reset_index()
df_1_resume_counts.columns = ['description', 'resume_count']
print(df_1_resume_counts.shape)
df_1_resume_counts

(1714, 2)


,description,resume_count
0,**Job Title: Data Scientist**\n\n**Job Summary...,1
1,**Job Title: Database Administrator**\n\n**Job...,1
2,**Job Title: HR Specialist**\n\n**Job Summary:...,1
3,**Job Title: UI/UX Designer**\n\n**Job Summary...,1
4,**Job Title:** AI Engineer\n\n**Job Summary:**...,1
...,...,...
1709,We're seeking a talented UI Engineer to work o...,7
1710,We're seeking a talented UI Engineer to work o...,5
1711,We're seeking a talented UX Designer to work o...,5
1712,We're seeking a talented UX Designer to work o...,5


In [12]:
# and of these positions, how many have both accepted and rejected resumes?
df_1_dual_decision = df_1.groupby('description')['decision'].nunique().reset_index()
df_1_dual_decision.columns = ['description', 'decision_types']
df_1_dual_decision = pd.merge(
    df_1_dual_decision, df_1_resume_counts,
    on='description', how='left'
)
df_1_dual_decision = df_1_dual_decision[df_1_dual_decision['decision_types'] == 2]
print(df_1_dual_decision.shape)
df_1_dual_decision

(864, 3)


,description,decision_types,resume_count
66,"As a AI Researcher, you will play a pivotal ro...",2,8
67,"As a AI Researcher, you will play a pivotal ro...",2,6
68,"As a AI Researcher, you'll lead the design and...",2,4
69,"As a AI Researcher, you'll lead the design and...",2,5
70,"As a AI Researcher, you'll lead the design and...",2,4
...,...,...,...
1708,We're seeking a talented UI Engineer to work o...,2,2
1709,We're seeking a talented UI Engineer to work o...,2,7
1710,We're seeking a talented UI Engineer to work o...,2,5
1712,We're seeking a talented UX Designer to work o...,2,5


In [13]:
# Narrow the dataset down to positions with both accepts and rejects
df_1_subset = df_1[df_1['description'].isin(df_1_dual_decision['description'])]
df_1_subset = df_1_subset.reset_index(drop=True)
print(df_1_subset.shape)
df_1_subset.head()

(4861, 5)


,role,resume,decision,reasoning,description
0,E-commerce Specialist,Jason Jones\nE-commerce Specialist\n\nContact ...,False,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,E-commerce Specialist,Patricia Gray\nContact Information:\n\n* Email...,True,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
2,E-commerce Specialist,Amanda Gross\nContact Information:\n\n* Email:...,False,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
3,Mobile App Developer,Jose Hall\nContact Information:\n\n* Email: [j...,False,No experience in back-end development.,We need a Mobile App Developer to enhance our ...
4,Cloud Engineer,Jessica Hall\nCloud Engineer\n\nContact Inform...,False,Needs improvement in machine learning algorithms.,We're seeking a talented Cloud Engineer to wor...


In [14]:
# Sample a job opening from the entire dataset, and show all its applications
random_state = 0
df_1_onejob = df_1_subset[df_1_subset['description'] == df_1_subset['description'].sample(random_state = random_state).item()]
print(df_1_onejob['description'].iloc[0])
df_1_onejob

As a System Administrator, you will play a pivotal role in shaping the future of e-commerce.


,role,resume,decision,reasoning,description
311,System Administrator,Christina Davis\nSystem Administrator\n\nConta...,False,Needs improvement in machine learning algorithms.,"As a System Administrator, you will play a piv..."
915,System Administrator,Mary Johnson\nSystem Administrator\n\nContact ...,False,Insufficient system design expertise for senio...,"As a System Administrator, you will play a piv..."
2133,System Administrator,John Irwin\nSystem Administrator\n\nContact In...,True,Impressive leadership and communication abilit...,"As a System Administrator, you will play a piv..."
3202,System Administrator,Travis Dawson\nSystem Administrator\n\nContact...,True,Perfectly aligned with data engineering needs.,"As a System Administrator, you will play a piv..."
3827,System Administrator,Marvin Bates\nSystem Administrator\n\nContact ...,True,Excellent full-stack development experience.,"As a System Administrator, you will play a piv..."
4105,System Administrator,Matthew Moreno\nSystem Administrator Candidate...,False,Lacked leadership skills for a senior position.,"As a System Administrator, you will play a piv..."
4258,System Administrator,Jerry Nelson\nSystem Administrator Candidate\n...,True,Strong technical skills in AI and ML.,"As a System Administrator, you will play a piv..."


In [15]:
# and sample a random resume 
print(df_1_onejob['resume'].sample().item())

Marvin Bates
System Administrator

Contact Information:

* Address: 123 Main St, Anytown, USA 12345
* Phone: (123) 456-7890
* Email: [marvin.bates@email.com](mailto:marvin.bates@email.com)
* LinkedIn: linkedin.com/in/marvinbates

Summary:
Highly skilled and experienced System Administrator with 8+ years of experience in managing and maintaining complex IT infrastructures. Proven track record of ensuring high availability, security, and performance of systems, networks, and applications. Skilled in Linux/Unix, Windows Server, Networking, Virtualization, and System Monitoring.

Technical Skills:

* Operating Systems: Linux (Red Hat, Ubuntu), Unix (Solaris, AIX), Windows Server (2008, 2012, 2016)
* Networking: Cisco, Juniper, VMware, Linux Networking (TCP/IP, DNS, DHCP)
* Virtualization: VMware vSphere, Hyper-V
* System Monitoring: Nagios, SolarWinds, Prometheus
* Scripting: Bash, Python, PowerShell
* Database Management: MySQL, Oracle

Professional Experience:

System Administrator, ABC 

In [16]:
# Build our pairwise format: select one good and one bad resume for each position
df_1_structured = (
    df_1_subset
    .groupby(['description', 'decision'], group_keys=False)
    .apply(lambda df: df.sample(n=1, random_state=0))
    .reset_index(drop=True)
)
df_1_structured.head()

C:\Users\dyaom\AppData\Local\Temp\ipykernel_9444\4050175593.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(n=1, random_state=0))


,role,resume,decision,reasoning,description
0,AI Researcher,Erin Kelly\nContact Information:\n\n* Email: [...,False,Needs improvement in machine learning algorithms.,"As a AI Researcher, you will play a pivotal ro..."
1,AI Researcher,Jason Foster\nAI Researcher\n\nContact Informa...,True,Excellent full-stack development experience.,"As a AI Researcher, you will play a pivotal ro..."
2,AI Researcher,Darlene Williams\nContact Information:\n\n* Em...,False,No experience in back-end development.,"As a AI Researcher, you will play a pivotal ro..."
3,AI Researcher,Melanie Livingston\nAI Researcher\n\nContact I...,True,Impressive leadership and communication abilit...,"As a AI Researcher, you will play a pivotal ro..."
4,AI Researcher,John Yang\nAI Researcher\n\nContact Informatio...,False,Lacks hands-on experience with cloud platforms.,"As a AI Researcher, you'll lead the design and..."


In [17]:
df_1_structured = (df_1_structured
                   .pivot(index='description', columns='decision', values='resume')
                   .rename(columns={False: 'resume_reject', True: 'resume_accept'})
                   .reset_index()
)
df_1_structured = df_1_structured[['description', 'resume_accept', 'resume_reject']]
df_1_structured

decision,description,resume_accept,resume_reject
0,"As a AI Researcher, you will play a pivotal ro...",Jason Foster\nAI Researcher\n\nContact Informa...,Erin Kelly\nContact Information:\n\n* Email: [...
1,"As a AI Researcher, you will play a pivotal ro...",Melanie Livingston\nAI Researcher\n\nContact I...,Darlene Williams\nContact Information:\n\n* Em...
2,"As a AI Researcher, you'll lead the design and...",Howard Carter\nContact Information:\n\n* Email...,John Yang\nAI Researcher\n\nContact Informatio...
3,"As a AI Researcher, you'll lead the design and...",Amber Henderson\nAI Researcher\n\nContact Info...,Andrew Hicks\nContact Information:\n\n* Email:...
4,"As a AI Researcher, you'll lead the design and...",Jennifer Sanford\nAI Researcher\n\nContact Inf...,Lisa Flores\nContact Information:\n\n* Email: ...
...,...,...,...
859,We're seeking a talented UI Engineer to work o...,Sara King\nContact Information:\n\n* Email: [s...,Brittany Webb\nUI Engineer\n\nContact Informat...
860,We're seeking a talented UI Engineer to work o...,Tracey Payne\nContact Information:\n\n* Email:...,"Emily Bullock, MD\nUI Engineer Candidate\n\nCo..."
861,We're seeking a talented UI Engineer to work o...,Jasmine Hall\nContact Information:\n\n* Email:...,William Castro\nContact Information:\n\n* Emai...
862,We're seeking a talented UX Designer to work o...,Gabriel Hall\nUX Designer\n\nContact Informati...,Nicole Andrade\nContact Information:\n\n* Emai...


### Dataset 2

https://huggingface.co/datasets/cnamuangtoun/resume-job-description-fit

In [18]:
# params

# What defines a good resume?
GOOD_RESUME_LABEL = "Good Fit"
BAD_RESUME_LABEL = "No Fit"

In [19]:
df_2_1_raw = pd.read_csv(dataset_2_url_1)
df_2_2_raw = pd.read_csv(dataset_2_url_2)
df_2_raw = pd.concat((df_2_1_raw, df_2_2_raw)).reset_index(drop=True)
df_2_raw.head()

,resume_text,job_description_text,label
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,No Fit
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,No Fit
2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,No Fit
3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",No Fit
4,SummaryWith extensive experience in business/r...,Life at Capgemini\nCapgemini supports all aspe...,No Fit


In [20]:
# no need for extra pre-processing the data here
df_2 = df_2_raw.copy()
print(df_2['resume_text'].sample().item())

SummaryTo participate as a team member in a dynamic work environment focused on promoting business growth and building
University popularity by providing superior value and service. Effective as a team member and as an individual. Experienced customer service representative          Experience with Oracle PeopleSoft SIS with excellent communication skills          Highly proficient in Slate CRM systems Enthusiastic, resourceful and enjoy challenges          Passionate about prospective student access Capable of effectively communicating at          to opportunity various levels with individuals and groups
Core QualificationsGuest servicesInventory control proceduresMerchandising expertiseLoss preventionCash register operationsProduct promotions
Professional Experience08/2014toCurrentData ProcessorAltus Group Limited–Dallas,TX,Process and verify prospective student application materials.Ensure data integrity for accurate dissemination among all offices of enrollment services.Manage emai

In [21]:
# Filter for job descriptions with both "Good fit" and "No fit"
descriptions = (df_2
                .groupby('job_description_text')['label']
                .apply(lambda x: {GOOD_RESUME_LABEL, BAD_RESUME_LABEL} <= set(x))
)
descriptions = descriptions[descriptions].index

len(descriptions), np.random.choice(descriptions, size=5)

(143,
 array(["Calling all innovators  find your future at Fiserv.\nWere Fiserv, a global leader in Fintech and payments, and we move money and information in a way that moves the world. We connect financial institutions, corporations, merchants, and consumers to one another millions of times a day  quickly, reliably, and securely. Any time you swipe your credit card, pay through a mobile app, or withdraw money from the bank, were involved. If you want to make an impact on a global scale, come make a difference at Fiserv.\nWhat does a successful Software Engineering Manager do at Fiserv?Fiserv is looking for a Software Engineering Manager to join our Core Acquiring Front End engineering group within Fiservs Global Business Solutions (Payment Acceptance) business.You will be joining an organization responsible for mission critical platforms that support our payments acquiring business. We process over 350 million payment transactions per day with a peak throughput of over 8,000 transact

In [22]:
# Build the structured dataset from dict
new_data_2 = [
    {
        'description': d,
        'resume_accept': df_2[
            (df_2['job_description_text']==d) & (df_2['label']==GOOD_RESUME_LABEL)
        ]['resume_text'].sample(random_state=0).item(),
        'resume_reject': df_2[
            (df_2['job_description_text']==d) & (df_2['label']==BAD_RESUME_LABEL)
        ]['resume_text'].sample(random_state=0).item(),
    } for d in descriptions
]
df_2_structured = pd.DataFrame(new_data_2)
df_2_structured

,description,resume_accept,resume_reject
0,\n\nPosition Title: Senior Construction Accoun...,Executive ProfileCapable Accountant successful...,SummarySkilled in high profile account managem...
1,\n\nResponsibilitiesLead and provide day-to-da...,SummarySoftware Developer with overall 3+ year...,Professional SummaryTo achieve a responsible p...
2,Experienced in Salesforce Industries Communic...,Career OverviewSenior data analyst with 8+ yea...,Professional SummaryExperienced Data Analyst c...
3,Lead Software Developer New York City - Hybr...,SummaryI am a computer engineer with over 12 y...,SummaryDynamic and motivated IT professional w...
4,"""A Little About This Gig""Agility Partners is s...","ProfileLogical Data Analyst, skilled in elicit...",SummaryQuality-focused Data Entry Clerk experi...
...,...,...,...
138,Who we are:Multi Media LLC is a technology com...,Professional SummaryYoung professional with\nh...,Professional SummaryWith the attitude of learn...
139,"Who we are?For the past 20 years, we have powe...",Professional SummaryHighly motivated Sales Ass...,"SummaryA results driven, detail-oriented data ..."
140,X is Alphabets moonshot factory. We are a dive...,Career Overview4+ years of experience in the f...,Professional SummaryWith the attitude of learn...
141,X is Alphabets moonshot factory. We are a dive...,Professional SummaryHighly skilled software de...,Professional SummaryElectrical Engineer with t...


### Dataset 3

https://huggingface.co/datasets/MikePfunk28/resume-training-dataset

In [23]:
if HAS_TOKEN:
    df_3_raw = pd.read_json(dataset_3_url, lines=True)
else:
    df_3_raw = pd.DataFrame()
df_3_raw.head()

,messages
0,"[{'role': 'system', 'content': 'You are an exp..."
1,"[{'role': 'system', 'content': 'You are an exp..."
2,"[{'role': 'system', 'content': 'You are an exp..."
3,"[{'role': 'system', 'content': 'You are an exp..."
4,"[{'role': 'system', 'content': 'You are an exp..."


In [24]:
df_3 = df_3_raw.copy()
if HAS_TOKEN:
        df_3 = pd.DataFrame(df_3['messages'].apply(lambda lst: {dct['role']: dct['content'] for dct in lst}).tolist())
print(df_3.shape)
df_3.head()

(22855, 3)


,system,user,assistant
0,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nGENE...,The candidate is a highly motivated General Ma...
1,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nDIRE...,The candidate is a seasoned Certified Manageme...
2,You are an expert resume assistant. You help u...,What job category does this resume best fit?\n...,This resume best fits the TEACHER category.
3,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nStep...,Stephanie Nelson is an experienced AI research...
4,You are an expert resume assistant. You help u...,Please summarize the following resume:\n\nDani...,Danielle Barnes is a results-driven digital ma...


In [25]:
if HAS_TOKEN:
    # System prompt is a bit useless. Drop it.
    unique_sys = df_3['system'].unique()
    print(unique_sys)
    assert len(unique_sys) == 1

    delim = "\n\n"
    df_3[['user_prompt', 'user_resume']] = df_3['user'].str.split(delim, n=1, expand=True)
    df_3 = df_3[['user_resume', 'user_prompt', 'assistant']] # dropped system prompt
print(df_3.shape)
df_3.head()

['You are an expert resume assistant. You help users write, critique, and improve their resumes to land their dream job.']
(22855, 3)


,user_resume,user_prompt,assistant
0,GENERAL MANAGER/FITNESS DIRECTOR Executive Pro...,Please summarize the following resume:,The candidate is a highly motivated General Ma...
1,DIRECTOR OF FINANCE Summary Seasoned Certified...,Please summarize the following resume:,The candidate is a seasoned Certified Manageme...
2,"TEACHER Summary Accomplished, exper...",What job category does this resume best fit?,This resume best fits the TEACHER category.
3,Stephanie Nelson AI Researcher Contact Informa...,Please summarize the following resume:,Stephanie Nelson is an experienced AI research...
4,Danielle Barnes Contact Information: * Email: ...,Please summarize the following resume:,Danielle Barnes is a results-driven digital ma...


In [26]:
if HAS_TOKEN:
    print(df_3['user_resume'].sample().item())

Ian Hatfield Contact Information: * Email: [ian.hatfield@email.com](mailto:ian.hatfield@email.com) * Phone: (555) 123-4567 * LinkedIn: linkedin.com/in/ianhatfield Professional Summary: Highly motivated and detail-oriented Cybersecurity Analyst with 5+ years of experience in risk assessment, network security, SIEM, and incident response. Proven track record of identifying and mitigating security threats, with a strong background in threat intelligence and analytics. Possess excellent communication and problem-solving skills, with the ability to work effectively in fast-paced and dynamic environments. Technical Skills: * Risk Assessment and Management * Network Security (Firewalls, VPNs, IDS/IPS) * SIEM (Splunk, ELK) and Log Analysis * Incident Response and Threat Intelligence * Compliance and Regulatory Requirements (HIPAA, PCI-DSS) * Operating Systems (Windows, Linux, macOS) * Scripting languages (Python, PowerShell) * Familiarity with cloud security platforms (AWS, Azure) Professional

## Final Datasets

In [27]:
# Dataset 1
df_1.sample(10, random_state=0)

,role,resume,decision,reasoning,description
6566,E-commerce Specialist,Sonia Orozco\nE-commerce Specialist\n\nContact...,False,No experience in back-end development.,Be part of a passionate team at the forefront ...
2515,Product Manager,Patricia Martinez\nContact Information:\n\n* E...,True,Excellent full-stack development experience.,Join our team as a Product Manager and leverag...
998,UX Designer,William Mosley\nUX Designer\n\nContact Informa...,False,Insufficient system design expertise for senio...,We need a UX Designer to enhance our team's te...
4837,Machine Learning Engineer,Jacob Adams\nMachine Learning Engineer\n\nCont...,False,Lacked leadership skills for a senior position.,Help us build the next-generation products as ...
6099,Data Analyst,Desiree Smith\nContact Information:\n\n* Phone...,False,Lacks hands-on experience with cloud platforms.,We are looking for an experienced Data Analyst...
4369,Data Analyst,Heather Hogan\nContact Information:\n\n* Email...,False,Needs improvement in machine learning algorithms.,Help us build the next-generation products as ...
4741,Machine Learning Engineer,Autumn Garza\nContact Information:\n\n* Addres...,True,Solid experience in machine learning and AI.,We're seeking a talented Machine Learning Engi...
6141,IT Support Specialist,Matthew Clark\nContact Information:\n\n* Phone...,True,Perfectly aligned with data engineering needs.,We're seeking a talented IT Support Specialist...
569,Blockchain Developer,James Chase\nBlockchain Developer\n\nContact I...,False,Lacked leadership skills for a senior position.,Looking for an experienced Blockchain Develope...
298,Cloud Architect,Riley Bradford\nContact Information:\n\n* Emai...,True,Strong technical skills in AI and ML.,"If you're passionate about cloud technologies,..."


In [28]:
df_1_structured.sample(10, random_state=0)

decision,description,resume_accept,resume_reject
55,"As a Data Architect, you will play a pivotal r...",Spencer May\nContact Information:\n\n* Address...,Shelly Garrett\nContact Information:\n\n* Phon...
316,"If you're passionate about cloud technologies,...",Debbie Mendoza\nBlockchain Developer\n\nContac...,April Ellis\nBlockchain Developer\n\nContact I...
252,Be part of a passionate team at the forefront ...,Mitchell Livingston\nContact Information:\n\n*...,Shawn Sanchez\nContact Information:\n\n* Addre...
262,Help us build the next-generation products as ...,Michael Goodman\nContact Information:\n\n* Pho...,Michelle Byrd\nContact Information:\n\n* Email...
31,"As a Cloud Engineer, you will play a pivotal r...",Jennifer Sherman\nContact Information:\n\n* Ph...,Jessica Mcdonald\nCloud Engineer\n\nContact In...
367,If you're passionate about software engineerin...,Brooke Austin\nContact Information:\n\n* Email...,Matthew Savage\nData Scientist\n\nContact Info...
270,Help us build the next-generation products as ...,Lori Chavez\nData Scientist\n\nContact Informa...,John Johns\nContact Information:\n\n* Email: [...
312,"If you're passionate about AI research, we nee...",Gina Nixon\nSystem Administrator\n\nContact In...,Matthew Rogers\nSystem Administrator\n\nContac...
680,We need a Digital Marketing Specialist to enha...,Brittany Cox\nContact Information:\n\n* Phone:...,Riley Sanchez\nDigital Marketing Specialist\n\...
688,We need a IT Support Specialist to enhance our...,Justin Reynolds\nContact Information:\n\n* Add...,Megan Wilson\nContact Information:\n\n* Addres...


In [29]:
# Dataset 2
df_2.sample(10, random_state=0)

,resume_text,job_description_text,label
3069,"SummaryA results driven, detail-oriented data ...","Hello,Greetings from DevCare SolutionsI got an...",No Fit
1675,SummaryQuality-focused Data Entry Clerk experi...,Net2Source Inc. is an award-winning total work...,No Fit
6385,Career OverviewQuality focused SQL Data Analys...,The ideal candidate will be responsible for de...,No Fit
543,SummaryResults-oriented college graduate with ...,We are looking for an experienced engineer who...,No Fit
3213,Professional SummaryHighly motivated Sales Ass...,Kindly focus on the highlighted skill in the b...,Potential Fit
134,SummaryLogical Data Analyst skilled in require...,Job Purpose: Perform designated tasks in the a...,No Fit
5869,Experience11/2017toCurrentElectrical EngineerG...,"Hello, please look into the below mentioned ro...",Good Fit
1425,SummaryI am a Principal Electrical Engineer an...,I own a small landscaping business that needs ...,No Fit
1819,Professional SummaryHighly regarded self-start...,Skills - Cucumber BDD + Selenium UI Automation...,No Fit
7492,ProfileWith 11+ years of IT experience in Ent...,"Primary Location: Melbourne, Florida\nV-Soft C...",Potential Fit


In [30]:
df_2_structured.sample(10, random_state=0)

,description,resume_accept,resume_reject
45,Electrical EngineerNDI Engineering Company is ...,ProfileTo obtain the position of a Telecommuni...,Summary7 years of industry experience developi...
118,Senior Accountant\nOur client is a growing aut...,SummaryFinancial Accountant specializing in fi...,Professional SummaryManagement specialist dedi...
16,About Milliman\n\nIndependent for over 75 year...,Professional SummaryExperienced Data Analyst c...,Experience10/2019to06/2019Data AnalystAngi Hom...
56,Industry leading client is seeking a Senior Ac...,"SummaryHighly organized, detail-oriented, and ...",Career OverviewExperienced Data Analyst with e...
22,"As a Manager, you will be a key driver in buil...",Professional SummaryYoung professional with\nh...,Professional SummaryTo achieve a responsible p...
7,10627 Software Engineer II (Heroku Architect)...,Summary• \nOver\nThree years of extensi...,Professional SummaryTo achieve a responsible p...
108,Rightware is the company behind the Kanzi fami...,HighlightsProficient inObjective - CJavaC/C++P...,SummaryPursue a challenging and rewarding posi...
134,Want to learn more about this role and Jobot? ...,ProfileTo obtain the position of a Telecommuni...,QUALIFICATIONS• Over two years' experience o...
130,"This is a W2 contract in Minneapolis, MN 55402...",SummaryProactive BI Developer/Data Analyst wit...,Experience10/2019to06/2019Data AnalystAngi Hom...
101,Qualifications:Accounting degreeCPA license re...,SummarySkilled in high profile account managem...,Professional SummaryCustomer Service Represent...


In [31]:
# Dataset 3
df_3.sample(10, random_state=0)

,user_resume,user_prompt,assistant
18722,ACCOUNTANT III Senior Accountant Res...,What job category does this resume best fit?,This resume best fits the ACCOUNTANT category.
15478,INFORMATION TECHNOLOGY COORDINATOR Profe...,Rewrite and improve this resume for a INFORMAT...,Here's an improved version for a INFORMATION-T...
11686,Here is a professional resume for Brenda Horn:...,Please summarize the following resume:,Brenda Horn is a seasoned Cloud Architect with...
4691,here's a sample resume for steven frey applyin...,Please summarize the following resume:,Steven Frey is a skilled UI Engineer with over...
4653,"b'JOB SEEKER 128 Pleasant Street Minneapolis, ...",Please summarize the following resume:,The candidate holds a Master's in Education an...
13664,Elizabeth Richard Blockchain Developer Contact...,Please summarize the following resume:,Elizabeth Richard is a highly skilled blockcha...
13903,Gerald Morris Contact Information: * Address: ...,Please summarize the following resume:,Gerald Morris is an e-commerce professional wi...
4068,/\nNGUYỄN ĐÌNH DUẨN\nCHUYÊN VIÊN KINH DOANH\n...,Critique this resume and suggest improvements:,This resume could benefit from clearer formatt...
20442,Tấn Đạt Vũ Trương \nDEVOPS - 6 YEARS OF EXPERI...,Critique this resume and suggest improvements:,This resume could benefit from clearer formatt...
8380,INFORMATION TECHNOLOGY PROVISIONING TECHNICIAN...,What are the key skills and qualifications in ...,The key skills for this INFORMATION-TECHNOLOGY...


## Export

In [32]:
DATASET_PATH.mkdir(parents=True, exist_ok=True)

UNSTRUCTURED_PATH = DATASET_PATH / 'unstructured'
UNSTRUCTURED_PATH.mkdir(parents=True, exist_ok=True)

def write_dataset(name, df):
    df.to_csv(DATASET_PATH / (name + '.csv'), index=False)

def write_structured_datasets(name, df_structured, random_state=0):
    # write full, 50-subset, 5-subset outputs
    
    # full
    write_dataset(name+'_full', df_structured)

    # 50
    df_50 = df_structured.sample(50, random_state=random_state)
    write_dataset(name+'_subset_50', df_50)

    # 5
    df_5 = df_structured.sample(5, random_state=random_state)
    write_dataset(name+'_subset_5', df_5)


In [33]:
write_dataset('unstructured/'+dataset_1_name+'_unstructured', df_1)
write_structured_datasets(dataset_1_name, df_1_structured)

In [34]:
write_dataset('unstructured/'+dataset_2_name+'_unstructured', df_2)
write_structured_datasets(dataset_2_name, df_2_structured)